# HighRes MicroServe

```{device-card} highres-microserve
```

| Property | Value |
| --- | --- |
| Transport | Ethernet, TCP port 1000 |
| Storage | 14 stackers, indexed 0–13 |
| Direct connection address | `10.253.253.253` |
| Verified controller | HRB-2008-10558, firmware 2.7.0.756 |
| Public length units | millimeters |

**Plate transfers, counting, plate-dimension calculation, successful barcode scanning, and physical fault recovery remain unverified.** Communication, homing, all fourteen carousel positions, and empty receiving-position preparation/retraction, stack-height measurement, and manual access were checked on this controller. Laser command exchanges and barcode geometry-error handling were also captured. Review the physical setup before running each motion cell.

This driver prepares the MicroServe for a separate robot to place or pick plates. It does not model plate inventory in PyLabRobot resources. Reported plate counts are the controller's cached estimates.

See the [state machine](state-machine.md) for workflow and recovery diagrams, and [protocol and validation](protocol.md) for the device-source mapping and verification limits.

The [capability audit](capabilities.md) lists implemented operations, hardware evidence, and maintenance gaps.

## Connection and manufacturer documentation

Connect the powered MicroServe's Ethernet port to a dedicated NIC. Configure that NIC with `10.253.253.50/24`, then use `10.253.253.253` as the host. All HighRes controllers share that alternate address, so use it only on a dedicated link. Your normal network connection can remain separate.

With the controller connected, open its address in a browser and choose **Support Files → MicroServe_API.pdf**. This guide follows revision 756, dated May 29, 2020. See the [protocol reference](protocol.md) for the other documentation available on the controller. The saved network address shown by the web interface can differ from legacy in-memory `settings` values.

The protocol acknowledges a command with an identifier, sends data, then sends a completion with the same command and identifier. The driver handles framing and converts plate dimensions from millimeters to device micrometers.

## Physical setup

Secure the machine, keep the robot outside its motion area, and inspect the transfer position. Each stacker must contain only one compatible plate geometry. Measure plate height, stacking pitch, and spatula support thickness before moving plates. Resolve an active E-stop through the machine's normal procedure.

## Connect

`setup()` opens the socket without homing, clearing errors, or changing calibration.

In [ ]:
from pylabrobot.high_res import HighResMicroServe, MicroServePlateDimensions

microserve = HighResMicroServe(host="10.253.253.253")
await microserve.setup()

## Identification

Read the controller identification report.

In [ ]:
print(await microserve.request_version())

## Firmware version

Read the machine-readable firmware version.

In [ ]:
await microserve.request_firmware_version()

## Status

Inspect homing, selected stacker, loader state, and the plate-detection beam. `plate_sensor_blocked` reports the beam signal; it does not establish plate occupancy at the virtual transfer position. An unknown stacker before homing is `None`.

Use `status.loader_retracted` and `status.loader_extended` to check the loader sensors. The raw `status.loader` field can stay `"extended"` after retraction on this firmware.

In [ ]:
status = await microserve.request_status()
status

## Readiness

This reports readiness to present a plate to the robot. It is false while the loader is retracted, even when the machine is homed and idle. Use `request_status()` to check homing and busy state.

In [ ]:
await microserve.is_ready()

## Error log

Read error entries without clearing them. Their numbers identify log entries, not stable fault codes.

In [ ]:
await microserve.request_errors()

## Settings

Read the current in-memory settings without changing or saving them.

In [ ]:
settings = await microserve.request_settings()
settings["PRODUCT_NAME"]

## Current plate geometry

Read all stackers' configured dimensions in millimeters.

In [ ]:
await microserve.request_dimensions()

## Cached plate counts

These are approximate firmware counts. This query does not move the carousel or physically recount plates.

In [ ]:
await microserve.request_plate_counts()

## Detailed hardware versions

Read firmware checksums and motor-controller versions.

In [ ]:
print(await microserve.request_detailed_version())

## Motor identities

Read the serial number and configured name of each axis controller.

In [ ]:
await microserve.request_motor_information()

## Command history

Read the latest three command history entries without repeating any command.

In [ ]:
await microserve.request_history(3)

## Command completion record

Query the last acknowledged command. IDs are only valid until the controller restarts.

In [ ]:
command_id = microserve.last_command_id
assert command_id is not None
await microserve.request_command_status(command_id)

## One stacker

Read a single stacker's cached approximate count without measurement motion.

In [ ]:
await microserve.stackers[0].request_plate_count()

## Home

**Moves hardware.** Clear the robot and transfer position first. Homing is explicit. A repeated call checks status and does not home again when already homed.

In [ ]:
await microserve.home()

## Plate geometry

Use measurements for the plates in this stacker. The values below are examples, not a plate-type recommendation:

- `height`: bottom of plate to top.
- `stack_height`: bottom-to-bottom distance between adjacent stacked plates.
- `thickness`: top of plate to the underside of its wells where the spatula supports it.

Each prepare or scan call receives geometry explicitly.

In [ ]:
dimensions = MicroServePlateDimensions(height=11.0, stack_height=10.0, thickness=10.0)
stacker = microserve.stackers[0]

## Set dimensions

Apply geometry for this stacker and verify it by reading it back. Preparation and scan methods also perform this step.

In [ ]:
await stacker.set_dimensions(dimensions)

## Rotate to a stacker

**Moves hardware.** The loader must be retracted and the transfer position clear.

In [ ]:
await stacker.move_to()

## Prepare to load a plate

**Moves hardware.** Present an empty receiving position for the robot. This command does not move the robot. Repeating it in the same verified state does not move the loader again.

In [ ]:
await stacker.prepare_for_load(dimensions)

## Finish loading

Have the robot place the plate at the taught transfer position, then withdraw completely. Only after that physical action, retract the mechanism.

In [ ]:
await microserve.retract()

## Prepare to unload a plate

**Moves hardware.** Present a plate from this stacker for the robot to pick. The method confirms the command completed and the selected stacker and loader reached the expected state. A repeated preparation owned by this driver does not fetch another plate before retraction, even if the beam signal changes. Actual robot pickup geometry is unverified.

In [ ]:
await stacker.prepare_for_unload(dimensions)

## Finish unloading

Have the robot pick the presented plate and withdraw completely before running this cell.

In [ ]:
await microserve.retract()

## Scan stacker barcodes

**Moves hardware.** The robot must be clear and the loader retracted. The returned tuple contains the controller's unmodified barcode data lines; barcode formatting has not yet been verified on this machine.

In [ ]:
await stacker.scan_barcodes(dimensions)

## Firmware diagnostics

The compact state, limit, angle, and home-offset reports retain device text. Axis limits are implausible on firmware 2.7.0.756; do not use them for travel bounds. Angle and home-offset units are undocumented.

In [ ]:
print(await microserve.request_variable_status())
print(await microserve.request_limits())
print(await microserve.request_plate_angle())
print(await microserve.request_home_offset(1))

## Firmware command help

Read command descriptions without executing those commands.

In [ ]:
print("\n".join(await microserve.request_command_help("measurestacker")))
print("\n".join(await microserve.request_command_catalog()))

## Measure stack height

Keep the robot clear. This moves the loader to measure the stack relative to its calibrated beam and returns millimeters. A single upright plate measures its well thickness. Small negative empty-stack readings are retained for calibration diagnosis. The tested firmware leaves the loader extended after measurement.

In [ ]:
height = await stacker.measure_height()
print(height)

## Retract after measurement

Inspect the measured stack and clear the robot before retraction. This must precede another stacker operation.

In [ ]:
await microserve.retract()

## Actively count plates

With correct plate geometry, measure this stacker and read the refreshed approximate count. This operation moves hardware; cached count queries do not. Inspect the loader afterward.

In [ ]:
count = await stacker.count_plates(dimensions)
print(count)

## Retract after counting

Clear the robot and retract before another stacker operation.

In [ ]:
await microserve.retract()

## Correct a cached count

After physically inspecting an empty stacker, set its bookkeeping count to zero. This does not remove plates or measure occupancy. Supply the observed count for a nonempty stacker.

In [ ]:
await stacker.set_plate_count(0)

## Enter manual mode

Finish any handoff and retract first. With the robot clear, release the carousel for manual access. Repeating the call in verified manual mode does not send another motion command.

In [ ]:
await microserve.enter_manual_mode()

## Return from manual access

Finish manual access, remove your hands, and clear the robot before homing.

In [ ]:
await microserve.home()

## Enable the barcode laser

The absolute on/off command does not move the scanner axis. Observe the laser state without looking into its beam; the firmware has no laser-state query.

In [ ]:
await microserve.set_barcode_laser(True)

## Disable the barcode laser

Turn the laser off after inspection.

In [ ]:
await microserve.set_barcode_laser(False)

## Measure plate dimensions

This requires the separate four-stacker arrangement described in the [validation procedure](validation.md): stacker 1 empty, an inverted plate in 2, an upright plate in 3, and at least three plates in 4. Confirm physical labels against indices and clear the robot. Inspect the returned dimensions before using them.

In [ ]:
measured_dimensions = await microserve.calculate_plate_dimensions(count=3)
print(measured_dimensions)

## Retract after dimension measurement

Inspect the arrangement, clear the robot, and retract before changing stackers or geometry.

In [ ]:
await microserve.retract()

## Prepare an unload with angle reporting

This presents a plate for pickup and returns the firmware angle as text. The angle unit is undocumented. Repeating this preparation queries the cached angle without fetching another plate.

In [ ]:
angle_report = await stacker.prepare_for_unload_with_angle(dimensions)
print(angle_report)

## Finish the angle-reporting handoff

Pick the presented plate and withdraw the robot before retracting.

In [ ]:
await microserve.retract()

## Clear an abort

Use only after resolving the abort cause and clearing the robot. This clears a firmware latch; it neither homes nor resolves an uncertain plate handoff.

In [ ]:
await microserve.clear_abort()

## Recover from an E-stop

Inspect plate support, release the E-stop according to the manufacturer procedure, and clear the robot before this explicit recovery operation. It can move hardware. Do not race firmware automatic recovery. An unresolved handoff stays unresolved.

In [ ]:
await microserve.recover_from_estop()

## Failures and reconnecting

`MicroServeError` preserves the command, identifier, completion status, and diagnostic lines for `ERROR`, `ABORTED`, and `WARNING` replies. All three interrupt the operation.

A timeout, cancelled task, or malformed reply closes the connection. The driver never automatically repeats a command. Closing the socket does not stop motion already executing. Inspect the physical machine before reconnecting with `setup()` and reading status.

An interrupted load/unload is retained in `unresolved_preparation`, blocking further preparation. The beam signal alone cannot identify which transfer completed. If the controller has not rebooted and no other client has moved it, `reconcile_preparation()` reads the original command record and current position to confirm success. Failed or unidentifiable commands stay unresolved. Explicit retraction ends a handoff only after the operator has inspected it and the robot is clear.

Failed measurements and barcode scans retain `unresolved_operation`. This also survives reconnection and blocks fresh motion. After inspection, explicit `retract()` clears it only when retraction is confirmed.

## Inspect an interrupted preparation

This is `None` when no load/unload is unresolved.

In [ ]:
microserve.unresolved_preparation

## Reconcile without moving

Use only after inspecting the machine and confirming it has not rebooted or been moved by another client. This reads the command record and loader state; it is a no-op when nothing is unresolved.

In [ ]:
await microserve.reconcile_preparation()

## Disconnect

Close the connection without homing, retracting, aborting, or clearing errors.

In [ ]:
await microserve.stop()